# Lab 25: AI Literacy & Generative Tools — Diagnostic Lab (ECON 5200)
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 70 min Core + 65 min AI (+ 20 min before the lab)

---

**Format:** Part 1 builds and deploys a portfolio site with Claude Code. Part 2 gives you a
retrieval-augmented generation (RAG) pipeline over excerpts from the 2023 FOMC minutes, written in
plain numpy and scikit-learn, that runs without an error and retrieves the wrong evidence: you
diagnose it, fix it, and then write its retrieval function yourself. The AI-Assisted Expansion puts AI to work three ways, each checked by
you: it writes the function you just wrote, it answers two economic questions with and without the
sources, and it hands you an analysis to check against them.

**Learning Objectives:**
- Deploy a portfolio website built with an AI coding tool, and check every number on it against your own work
- Build each step of a RAG pipeline in plain code: chunk the documents, turn the chunks into TF-IDF vectors, retrieve the most similar ones by cosine similarity, and put them into a prompt
- Diagnose chunking, vocabulary and retrieval errors from what a pipeline prints
- Prompt an AI to write a function you have already written, and compare its numbers with yours
- Judge which of an AI's answers the sources support, with retrieval and without, and correct an AI-written analysis against the sources

**Tools:** Claude Code, Next.js, GitHub and Vercel (Part 1); numpy, scikit-learn and Colab's
built-in Gemini, through `google.colab.ai` (Part 2 and the AI-Assisted Expansion). No API key and no
paid account.

**Deliverable:** your live portfolio URL and its repository; your diagnoses and the fixed pipeline
(Part 2); the required `top_k_chunks()` function; and the AI-Assisted Expansion: your prompts, the
recorded replies and your judgments, the critique table, the checking log and two answers.

**Time:** Core (70 min: Part 1 15, Part 2 35, Write it yourself 20) + AI-Assisted Expansion
(required, 65 min)

With the Final Reflection and the submission, plan on about two hours and 25 minutes, most of it
after class. Before this lab, a week early, the installs for Part 1 take about 20 minutes more.

**AI in this lab:** three rules, one per stage. **Part 1** is built with Claude Code, and AI is
required: use Northeastern's Claude (claude.northeastern.edu, and Claude Code signed in with your
Northeastern account), keep every prompt you send, and be able to explain every line of the site you
deploy. **Part 2 and Write it yourself** follow the usual rule: the diagnoses, the fixes and
`top_k_chunks` are yours. You may ask Colab to *explain* an error or a line (click **Explain
error**); do not ask any AI to find the errors or to write or fix the code, and turn off AI code
completion while you work on them. **The AI-Assisted Expansion** is where AI writes and answers, and
you write every prompt and check every reply.

---

> ### What you write, and what you run
>
> **You build:** the portfolio site in Part 1, by directing Claude Code: your bio and one project, with your own numbers. The installs it needs are in **Before this lab**.
>
> **You write code:**
> - **Part 2:** four blanks in the analyst's code, your fix for what your diagnosis found. The lab then runs its own fixed pipeline, so that everyone's later numbers match.
> - **Write it yourself (required, after Part 2):** `top_k_chunks()`, retrieval by cosine similarity, about six lines in an empty cell. A check cell tells you when it is right.
> - **AI-Assisted Expansion:** a reply pasted into `PASTED` wherever Colab's AI could not answer for you, and your judgments in `JUDGED`. You type entries into both; there is no code to write there.
>
> **You write, in text cells:** your live URL and two lines on Part 1; three diagnoses in Part 2; in the AI-Assisted Expansion, your own prompt for `ai_top_k_chunks` (and a revised one if you need a second round), a short comparison, a four-row critique table, the checking log and two answers (Critique and Judgment); and one Final Reflection answer (R1). Your prompts, judgments, critique and log are graded, not the AI's code, which you paste rather than type.
>
> **You run and read:** everything else. That is the FOMC excerpts, the chunker, the analyst's three cells, the lab's fixed pipeline, the cosine similarity by hand, the helper that sends a prompt to Colab's Gemini, the prompts the lab builds, the passages printed beside each reply, the AI-written analysis and the search helper. Read each cell before you run it: the questions ask about what it prints. The check cells: run them; you need not follow their code.

---

## Before this lab (do this a week early — about 20 min)

Part 1 needs these installs and accounts. They are not counted in the lab's time, and they take
longest the first time, so do them a week early. Every command below runs in a terminal on your computer (Terminal on a Mac, **PowerShell** on
Windows), not in Colab. Where the two differ, both are given.

1. **Node.js 18 or later**, which includes `npm`: install the LTS version from
   [nodejs.org](https://nodejs.org), then check with `node --version`.
2. **Git, with your name and email** (each commit is signed with them):
   ```
   git config --global user.name "Your Name"
   git config --global user.email "you@northeastern.edu"
   ```
3. **The GitHub CLI**, `gh`, from [cli.github.com](https://cli.github.com); then sign in with
   `gh auth login` (choose GitHub.com, HTTPS, and log in with a web browser).
4. **A Vercel account:** sign up at [vercel.com](https://vercel.com) with **Continue with GitHub**.
5. **Claude Code**, signed in with your Northeastern account:
   ```
   npm install -g @anthropic-ai/claude-code
   claude --version
   claude auth login
   claude auth status
   ```
   There is no `claude login`: Claude Code treats an unknown first word as a prompt, so
   `claude login` opens a chat about the word "login".
6. **The portfolio skill.** Download `build-portfolio-site.zip` from Canvas (Module 25). A skill is a
   folder on disk, so you unzip it where Claude Code looks for skills (there is no
   `claude skill install`). Run these in the folder that holds the zip:

   Mac or Linux:
   ```bash
   mkdir -p ~/.claude/skills
   unzip build-portfolio-site.zip -d ~/.claude/skills/
   ls ~/.claude/skills/build-portfolio-site/SKILL.md      # prints the path
   ```
   Windows PowerShell:
   ```powershell
   New-Item -ItemType Directory -Force "$HOME\.claude\skills"
   Expand-Archive build-portfolio-site.zip -DestinationPath "$HOME\.claude\skills"
   Test-Path "$HOME\.claude\skills\build-portfolio-site\SKILL.md"      # prints True
   ```

**If Claude Code will not install or sign in:** do not switch to a personal account. Use Claude in
the browser at [claude.northeastern.edu](https://claude.northeastern.edu): ask it for one file at a
time and save each file yourself, then deploy as in Part 1. Say so under Record it, and message the
instructor on Canvas the same day.


---

## Part 1: Portfolio Site (15 min)

You deploy a portfolio site by directing Claude Code: it writes the files from the skill's template,
and you supply a bio and one project, check the numbers and deploy. Do **Before this lab** first.

### Build (7 min)

```
mkdir econ-portfolio
cd econ-portfolio
claude
```

In the session, type `/build-portfolio-site`, then give Claude Code, in your own words: your name,
program and a two- or three-sentence bio; one project from this course, with a title, one or two
sentences on what you did and found, its GitHub link and the tools you used; and your email. Keep
the skill's layout as it is. Then ask it to install the dependencies and start the dev server, and
open `http://localhost:3000`.

> **Every number on the site goes out under your name, on a public URL.** Take each figure from the
> output your own notebook printed: open the lab, find the line, copy the number. Do not let Claude
> Code invent or round one, and claim only what the lab measured.

### Deploy (6 min)

Each line is a separate command, so the same lines work in PowerShell:

```
git init
git add .
git commit -m "Initial portfolio"
gh repo create econ-portfolio --public --source=. --push
```

Import the repository at [vercel.com/new](https://vercel.com/new) and click **Deploy**. Open the
live URL on a computer and on a phone, and click the project link.

### Record it (2 min)

In the text cell below: your live URL, the prompt you had to revise and why, and one thing Claude
Code got wrong that you fixed (a number, a link, a layout bug).

*Optional, after the lab:* more projects, a style you like from [stitch.withgoogle.com](https://stitch.withgoogle.com), or a section of your own.

*Part 1:*

**Live URL:**

1. The prompt I revised, and why:
2. What Claude Code got wrong, and how I fixed it:

---

## Part 2: RAG Pipeline — Diagnose, Fix, Extend (35 min)

A retrieval-augmented generation (RAG) pipeline answers a question from documents rather than from
a model's memory, in four steps: **chunk** the documents into short passages, **embed** each passage
as a vector, **retrieve** the passages whose vectors are most similar to the question's, and
**generate** an answer from a prompt that contains them. Production pipelines often embed with a
neural model and keep the vectors in a vector database. This one does each step in a few lines of
numpy and scikit-learn, with the TF-IDF vectors of Chapter 23, so that you can see every step.

An analyst built one over excerpts from the minutes of the eight FOMC meetings of 2023. It runs
without an error, and the evidence it retrieves is wrong. Run the analyst's three cells, one per
step, read what each one prints, and find what went wrong and where before you fix anything. Write it yourself, after this Part, turns the retrieval into a function, and the AI-Assisted
Expansion adds the last step, generation.

### Setup

The first line of the setup cell installs scikit-learn 1.8 or 1.9, the versions the expected
numbers in this lab were produced on (both print the same numbers). The cell prints your versions.
If it stops with an `ImportError`, use **Runtime → Restart session** and run this cell again.

**Reading the code.** `TfidfVectorizer` comes from Chapter 23: `.fit_transform(texts)` learns a
vocabulary and gives one row of TF-IDF weights per text; `.transform(texts)` gives rows for new
texts in that vocabulary. `cosine_similarity(a, b)` is the cosine of the angle between rows: 1 for
rows that point the same way, 0 for rows with no word in common.

In [ ]:
# GIVEN — run as-is
# Setup: the libraries Part 2 uses, and their versions
!pip install "scikit-learn>=1.8,<1.10" -q

import numpy as np
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print(f"numpy {np.__version__}, scikit-learn {sklearn.__version__} (this lab needs 1.8 or 1.9)")

**The sources.** The next cell holds excerpts from the Federal Reserve's published minutes of each
2023 meeting: the staff's latest unemployment and PCE inflation figures, a sentence or two from the
discussion, and the decision on the target range for the federal funds rate. `...` marks a cut. A
meeting is named by the month it ended in: the February 2023 meeting ran from January 31 to
February 1.

**Reading the code.** Each meeting is a dictionary with two entries, read as `m["meeting"]` and
`m["text"]`. `.split()` cuts a text into words at the spaces, and `sum(... for m in MINUTES_2023)`
adds up one number per meeting. `ch.isascii()` is True for a plain keyboard character; a text
pasted from a web page can carry broken characters such as `â€”` where a dash was, and the last
number counts any character that is not plain.

In [ ]:
# GIVEN — run as-is
# The sources: excerpts from the minutes of the eight FOMC meetings of 2023, one entry per meeting
MINUTES_2023 = [
    {"meeting": "February 2023",
     "text": "The unemployment rate moved back down to 3.5 percent in December. Total PCE price "
             "inflation was 5.0 percent over the 12 months ending in December, 1.1 percentage points "
             "lower than the October figure. Core PCE price inflation, which excludes changes in "
             "consumer energy prices and many consumer food prices, was 4.4 percent over the 12 months"
             " ending in December, down 0.7 percentage point from its October reading. Members agreed "
             "that inflation had eased somewhat but remained elevated. In support of these goals, "
             "members agreed to raise the target range for the federal funds rate to 4-1/2 to 4-3/4 "
             "percent. Members anticipated that ongoing increases in the target range would be "
             "appropriate in order to attain a stance of monetary policy that is sufficiently "
             "restrictive to return inflation to 2 percent over time."},
    {"meeting": "March 2023",
     "text": "The unemployment rate was little changed and stood at 3.6 percent in February. Total PCE"
             " price inflation was 5.4 percent over the 12 months ending in January, and core PCE "
             "price inflation ... was 4.7 percent. Participants assessed that the developments so far "
             "would likely lead to some weakening of credit conditions, as some banks were likely to "
             "tighten lending standards amid rising funding costs and increased concerns about "
             "liquidity. Participants noted that it was too early to assess with confidence the "
             "magnitude of the effect of a credit tightening on economic activity and inflation ... In"
             " support of these goals, members agreed to raise the target range for the federal funds "
             "rate to 4-3/4 to 5 percent. Given recent developments, members anticipated that some "
             "additional policy firming may be appropriate in order to attain a stance of monetary "
             "policy that is sufficiently restrictive to return inflation to 2 percent over time."},
    {"meeting": "May 2023",
     "text": "The pace of increases in total nonfarm payroll employment slowed in March but was still "
             "robust, and the unemployment rate ticked down to 3.5 percent. Total PCE price inflation "
             "was 4.2 percent over the 12 months ending in March, and core PCE price inflation ... was"
             " 4.6 percent; the total inflation measure was down markedly from its level in January, "
             "while the core measure was only slightly lower. Many participants commented that the "
             "recent developments in the banking sector had contributed to some tightening of lending "
             "standards beyond that which had occurred during previous quarters, especially among "
             "small and mid-sized banks. In support of these goals, the members agreed to raise the "
             "target range for the federal funds rate to 5 to 5-1/4 percent."},
    {"meeting": "June 2023",
     "text": "The unemployment rate edged up, on net, but was still at a low level of 3.7 percent in "
             "May. Total PCE price inflation was 4.4 percent over the 12 months ending in April, and "
             "core PCE price inflation was 4.7 percent ... Against this backdrop, and in consideration"
             " of the significant cumulative tightening in the stance of monetary policy and the lags "
             "with which policy affects economic activity and inflation, almost all participants "
             "judged it appropriate or acceptable to maintain the target range for the federal funds "
             "rate at 5 to 5-1/4 percent at this meeting. Members agreed that holding the target range"
             " steady at this meeting allowed the Committee to assess additional information and its "
             "implications for monetary policy."},
    {"meeting": "July 2023",
     "text": "The unemployment rate edged down to 3.6 percent in June, while the labor force "
             "participation rate and the employment-to-population ratio were both unchanged. Total PCE"
             " price inflation was 3.8 percent over the 12 months ending in May, and core PCE price "
             "inflation, which excludes changes in energy prices and many consumer food prices, was "
             "4.6 percent over the same period. Participants noted that consumer spending had recently"
             " exhibited considerable resilience ... In support of the Committee's objectives to "
             "achieve maximum employment and inflation at the rate of 2 percent over the longer run, "
             "members agreed to raise the target range for the federal funds rate to 5-1/4 to 5-1/2 "
             "percent. All members affirmed that they are strongly committed to returning inflation to"
             " their 2 percent objective."},
    {"meeting": "September 2023",
     "text": "Over July and August, the unemployment rate edged up, on net, and stood at 3.8 percent "
             "in August ... The total price index for personal consumption expenditures (PCE) "
             "increased 3.3 percent over the 12 months ending in July, and core PCE price inflation "
             "... was 4.2 percent over the same period; both total and core PCE price inflation were "
             "lower than a year earlier. Several participants commented that, with the policy rate "
             "likely at or near its peak, the focus of monetary policy decisions and communications "
             "should shift from how high to raise the policy rate to how long to hold the policy rate "
             "at restrictive levels. ... members agreed to maintain the target range for the federal "
             "funds rate at 5-1/4 to 5-1/2 percent."},
    {"meeting": "November 2023",
     "text": "Although total nonfarm payroll employment rose at a faster pace in September than in "
             "previous months, the unemployment rate was unchanged at 3.8 percent ... The price index "
             "for total personal consumption expenditures (PCE) increased 3.4 percent over the 12 "
             "months ending in September, while core PCE price inflation ... was 3.7 percent over the "
             "same period. Members concurred that the U.S. banking system was sound and resilient. "
             "They also agreed that tighter financial and credit conditions for households and "
             "businesses were likely to weigh on economic activity, hiring, and inflation but that the"
             " extent of these effects was uncertain. ... members agreed to maintain the target range "
             "for the federal funds rate at 5-1/4 to 5-1/2 percent."},
    {"meeting": "December 2023",
     "text": "The unemployment rate was little changed, on net, and stood at 3.7 percent in November "
             "... The price index for total personal consumption expenditures (PCE) increased 3.0 "
             "percent over the 12 months ending in October, while core PCE inflation ... was 3.5 "
             "percent over the same period ... In discussing the policy outlook, participants viewed "
             "the policy rate as likely at or near its peak for this tightening cycle, though they "
             "noted that the actual policy path will depend on how the economy evolves. Several "
             "participants remarked that the Committee's past policy actions were having their "
             "intended effect of helping to slow the growth of aggregate demand and cool labor market "
             "conditions. In their submitted projections, almost all participants indicated that, "
             "reflecting the improvements in their inflation outlooks, their baseline projections "
             "implied that a lower target range for the federal funds rate would be appropriate by the"
             " end of 2024. ... members agreed to maintain the target range for the federal funds rate"
             " at 5-1/4 to 5-1/2 percent."},
]

# Text copied from the web can arrive with broken characters ("â€”" where a dash was);
# these excerpts were typed in plain ASCII, so the count below should be 0
odd = sum(not ch.isascii() for m in MINUTES_2023 for ch in m["text"])
print(f"{len(MINUTES_2023)} meetings, {sum(len(m['text'].split()) for m in MINUTES_2023)} words, "
      f"{odd} characters outside plain ASCII")

**Expected output:** `8 meetings, 1065 words, 0 characters outside plain ASCII`.

### The analyst's chunks

**Reading the code.** `chunk_words` cuts a text into chunks of `size` words, each starting
`size - overlap` words after the last, so neighbouring chunks share `overlap` words.
`range(0, n, step)` counts 0, step, 2 × step, ... below n; `words[a:b]` takes words a up to, not
including, b; `" ".join(...)` glues words back with spaces. In `build_chunks`, the second `for`
runs inside the first, and each chunk starts with its meeting's name. The last line runs
`chunk_words` on a sentence of ten words.

In [ ]:
# GIVEN — run as-is
# Cut each meeting's text into chunks of words; neighbouring chunks share `overlap` words
def chunk_words(text, size, overlap):
    """The text in chunks of `size` words, each starting size - overlap words after the last."""
    words = text.split()
    step = size - overlap
    return [" ".join(words[start:start + size]) for start in range(0, len(words), step)
            if start == 0 or start + overlap < len(words)]


def build_chunks(size, overlap):
    """Every meeting's chunks, each starting with its meeting's name, in one list."""
    return [f"{m['meeting']}: {chunk}" for m in MINUTES_2023
            for chunk in chunk_words(m["text"], size, overlap)]


print(chunk_words("the Committee raised the target range by a quarter point", size=4, overlap=1))

**Expected output:** `['the Committee raised the', 'the target range by', 'by a quarter point']`:
chunks of four words, each sharing one word with the chunk before.

`np.mean([...])` is the mean of a list of numbers, here the number of words in each chunk.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# Chunks: the analyst's chunk size and overlap, and three neighbouring chunks of the March meeting
CHUNK_SIZE, OVERLAP = 8, 0
analyst_chunks = build_chunks(CHUNK_SIZE, OVERLAP)
print(f"{len(analyst_chunks)} chunks, {np.mean([len(c.split()) for c in analyst_chunks]):.1f} words "
      "each on average (with the meeting's name)")
for i in [21, 22, 23]:
    print(f"[{i}] {analyst_chunks[i]}")

**Expected output:** `135 chunks, 9.9 words each on average (with the meeting's name)`, then
chunks [21] to [23] of the March meeting: the end of one sentence and the start of the next.

### The analyst's vectors

**Reading the code.** `stop_words="english"` drops common words such as "the" before counting; for
any other option, `help(TfidfVectorizer)` prints the documentation. `.vocabulary_` is the
dictionary of words kept, and `w in d` asks whether `w` is one of them. `.build_analyzer()` gives
the function that cuts a text into words, so `analyzer(QUESTION)` lists the question's words as the
vectorizer sees them. `.shape` is a matrix's (rows, columns).

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# Vectors: one TF-IDF row per chunk, and the question's words that the vocabulary kept
QUESTION = "How did participants expect the banking developments to affect credit conditions?"
analyst_vectorizer = TfidfVectorizer(stop_words="english", max_features=40)
analyst_matrix = analyst_vectorizer.fit_transform(analyst_chunks)
analyzer = analyst_vectorizer.build_analyzer()
print(f"matrix: {analyst_matrix.shape[0]} chunks x {analyst_matrix.shape[1]} words")
print("the question's words:", analyzer(QUESTION))
print("of those, in the vocabulary:", [w for w in analyzer(QUESTION) if w in analyst_vectorizer.vocabulary_])

**Expected output:** a matrix of 135 chunks × 40 words; the question's words are `did`,
`participants`, `expect`, `banking`, `developments`, `affect`, `credit` and `conditions`, and only
`participants` is in the vocabulary.

### The analyst's retrieval

**Reading the code.** `cosine_similarity(q, matrix)[0]` is one similarity per chunk. `np.argsort(a)`
gives the positions that would put `a` in order, smallest first; `a[::-1]` reverses `a`, and
`a[:3]` keeps its first three entries. `.max()` is the largest entry and `.argmax()` its position.
The next cell shows `np.argsort` on a small array.

In [ ]:
# GIVEN — run as-is
# np.argsort on a small array: positions, from the smallest value to the largest
a = np.array([0.2, 0.9, 0.5, 0.0])
print("np.argsort(a):   ", np.argsort(a))
print("a[np.argsort(a)]:", a[np.argsort(a)])
print("a[::-1]:         ", a[::-1])

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# Retrieval: every chunk's similarity to the question, and the K chunks the analyst keeps
K = 3
sims = cosine_similarity(analyst_vectorizer.transform([QUESTION]), analyst_matrix)[0]
order = np.argsort(sims)[:K]
for i in order:
    print(f"[{i}] similarity {sims[i]:.3f}  {analyst_chunks[i]}")
print(f"largest similarity of any chunk: {sims.max():.3f}, chunk [{sims.argmax()}] "
      f"{analyst_chunks[sims.argmax()]}")

**Expected output:** three chunks with similarity 0.000, then `largest similarity of any chunk:
0.782, chunk [124] December 2023: how the economy evolves. Several participants remarked that`.

### YOUR DIAGNOSIS

1. **The chunks.** Read chunks [21] to [23]. What does a reader, or a generator handed one of them as evidence, get from it? What in the analyst's chunking produces this, and what would you change?
2. **The vectors.** Compare the question's words with the ones the vocabulary kept. Which words were dropped, what do they have in common, and which line of the analyst's code dropped them? What can a chunk's similarity to this question still measure?
3. **The retrieval.** Compare the similarities of the three chunks kept with the largest similarity of any chunk. Which line chose those three, and what does it do? Would the most similar chunk answer the question?

*Your answers here:*

### Fix the pipeline

Fill in the four blanks with your fix for what your diagnosis found: a chunk size and an overlap
that keep the minutes' sentences readable, a vocabulary setting, and an order for the retrieved
chunks. `help(TfidfVectorizer)` lists what each option accepts. The cell prints what your pipeline
retrieves for the same question.

In [ ]:
# YOUR TASK — fill in the four blanks (____)
my_chunks = build_chunks(size=____, overlap=____)
my_vectorizer = TfidfVectorizer(stop_words="english", max_features=____)
my_matrix = my_vectorizer.fit_transform(my_chunks)
print(f"{len(my_chunks)} chunks; matrix {my_matrix.shape[0]} x {my_matrix.shape[1]}")
print("the question's words in the vocabulary:",
      [w for w in my_vectorizer.build_analyzer()(QUESTION) if w in my_vectorizer.vocabulary_])

my_sims = cosine_similarity(my_vectorizer.transform([QUESTION]), my_matrix)[0]
my_order = np.argsort(my_sims)[____][:K]
for i in my_order:
    print(f"[{i}] similarity {my_sims[i]:.4f}  {my_chunks[i]}")

**Expected output:** your number of chunks and the size of your matrix, the question's words your
vocabulary keeps, and three chunks, most similar first, about banks and credit conditions. If the
three chunks still have similarity 0.0000, or the most similar is not first, look again at your
third diagnosis.

### The lab's fixed pipeline

So that everyone's numbers match from here on, the rest of the lab uses the fixed pipeline below:
chunks of 40 words that overlap by 10, every word the chunks contain (`max_features=None`), and the
most similar chunk first. Compare it with your fix: where you chose differently, say why in one
line under your diagnoses.

In [ ]:
# GIVEN — run as-is
# The fixed pipeline the rest of the lab uses: 40-word chunks overlapping by 10, the full vocabulary
chunks = build_chunks(size=40, overlap=10)
vectorizer = TfidfVectorizer(stop_words="english", max_features=None)
matrix = vectorizer.fit_transform(chunks)
print(f"{len(chunks)} chunks; matrix {matrix.shape[0]} x {matrix.shape[1]}")

sims = cosine_similarity(vectorizer.transform([QUESTION]), matrix)[0]
order = np.argsort(sims)[::-1][:K]
for i in order:
    print(f"[{i}] similarity {sims[i]:.4f}  {chunks[i]}")

**Expected output:** `36 chunks; matrix 36 x 222`, and three chunks, most similar first: [28]
(November, 0.3136), [6] (March, 0.3068) and [12] (May, 0.2318), the minutes' sentences on credit
conditions and the banking sector.

### Cosine similarity, by hand

Write it yourself asks you to compute the similarities yourself. The next cell does it for the
question and one chunk.

**Reading the code.** `.toarray()` turns the sparse matrix (it stores only non-zero weights) into an
ordinary array, one row per text. `q @ c` is the dot product: multiply entry by entry, then add up.
`np.linalg.norm(v)` is a vector's length; with `axis=1`, one length per row. `rows @ q` gives one
dot product per chunk, and `a[order]` picks the entries of `a` at the positions in `order`.

In [ ]:
# GIVEN — run as-is
# Cosine similarity by hand: the dot product of two vectors, divided by the product of their lengths
q = vectorizer.transform([QUESTION]).toarray()[0]      # the question: one weight per vocabulary word
rows = matrix.toarray()                                 # the chunks: one row per chunk
c = rows[6]
print(f"chunk [6]: by hand {(q @ c) / (np.linalg.norm(q) * np.linalg.norm(c)):.4f}, "
      f"cosine_similarity {sims[6]:.4f}")
print("lengths of the first five rows:", np.linalg.norm(rows, axis=1)[:5].round(4))
print("rows @ q, at the retrieved positions:      ", (rows @ q)[order].round(4))
print("cosine_similarity, at the same positions:  ", sims[order].round(4))

**Expected output:** `chunk [6]: by hand 0.3068, cosine_similarity 0.3068`; five lengths of 1;
and `[0.3136 0.3068 0.2318]` twice. `TfidfVectorizer` scales every row to a length of 1, the
question's too, so for these vectors the dot product already equals the cosine similarity. Dividing
by the lengths changes nothing here, and it keeps the function right for vectors that are not
scaled.

---

## Write it yourself (required — 20 min)

This function is required. No AI: write it yourself, with AI code completion turned off.

The lab's fixed pipeline retrieved for one question, with `cosine_similarity` and a fixed K. A pipeline needs
retrieval as a function it can call with any question, any vectorizer and matrix, and any number of
chunks. In the empty cell below, write a function `top_k_chunks(question, vectorizer, matrix, k=3)`
that:

1. turns `question` into a vector with `vectorizer`, in the vocabulary it has already learned (do
   not fit it again);
2. computes the cosine similarity of that vector with every row of `matrix` yourself, as the cell
   above did by hand, without calling `cosine_similarity`;
3. returns two numpy arrays: the positions (indices) of the `k` most similar chunks, most similar
   first, and their similarities, in the same order.

Use only the arguments: the check also calls the function with a second set of chunks, cut and
vectorized differently from this lab's. You need about six lines.

The check runs your function on the question above with k=3, where it should give back chunks
[28, 6, 12] with similarities 0.3136, 0.3068 and 0.2318, as the lab's pipeline printed; on two new
questions with k=5 and k=2; on 26 chunks of 60 words with k=4; and, with k=3, on this lab's chunks
vectorized with `TfidfVectorizer(norm=None)`, which leaves the rows unscaled. Until you write the function, the
check says "Not written yet"; once you do, it fails with a message until your function is right.

In [ ]:
# WRITE IT YOURSELF — required: write top_k_chunks(question, vectorizer, matrix, k=3)

In [ ]:
# CHECK — for top_k_chunks above. Run it; you need not read it.
# globals() holds every name defined so far in this notebook
if "top_k_chunks" not in globals():
    print("Not written yet: write top_k_chunks in the empty cell above, then run this check again.")
else:
    import copy
    import inspect

    def names_used(code):
        """Every global name a function's code refers to, including inside functions it defines."""
        names = set(code.co_names)
        for const in code.co_consts:
            if hasattr(const, "co_names"):
                names |= names_used(const)
        return names

    # inspect.signature lists the names of a function's arguments, in order, with their defaults
    signature = inspect.signature(top_k_chunks)
    params = list(signature.parameters)
    assert params[:4] == ["question", "vectorizer", "matrix", "k"], (
        f"top_k_chunks's arguments are ({', '.join(params)}). The check calls it as the task asks: "
        "top_k_chunks(question, vectorizer, matrix, k=3).")
    assert signature.parameters["k"].default == 3, (
        "Give k a default of 3: def top_k_chunks(question, vectorizer, matrix, k=3).")
    used = names_used(top_k_chunks.__code__)
    from sklearn.metrics import pairwise
    ready_made = {pairwise.cosine_similarity: "cosine_similarity", pairwise.linear_kernel: "linear_kernel"}
    for name in sorted(used):
        # "is" asks whether a name holds the very same function, under whatever name it was imported
        found = [label for func, label in ready_made.items() if globals().get(name) is func]
        assert name not in ("cosine_similarity", "linear_kernel") and not found, (
            f"top_k_chunks calls {found[0] if found else name}"
            f"{' (as ' + name + ')' if found and found[0] != name else ''}. Compute the similarity "
            "yourself, as the cell before Write it yourself did by hand.")
    assert "chunks" not in used and "analyst_chunks" not in used, (
        "top_k_chunks uses chunks, the lab's list of texts. Give back the positions (indices) of the "
        "chunks, not their text, and compute everything from the arguments.")
    import scipy.sparse
    for lab_name in sorted(used):
        value = globals().get(lab_name)
        lab_data = (isinstance(value, (np.ndarray, list, tuple, dict, str, int, float))
                    or scipy.sparse.issparse(value) or hasattr(value, "vocabulary_"))
        assert not (lab_data and not isinstance(value, bool)), (
            f"top_k_chunks uses {lab_name}, which the lab defined ({type(value).__name__}), instead "
            "of its arguments. Compute everything from question, vectorizer, matrix and k.")

    # The lab's fixed pipeline
    try:
        assert matrix.shape == (len(chunks), len(vectorizer.vocabulary_))
    except Exception:
        raise AssertionError(
            "chunks, vectorizer and matrix are not what the lab's fixed pipeline made. Run that "
            "cell again, then this check.") from None

    def lab_top_k(question, vec, mat, k):
        """The fixed pipeline's retrieval, to compare with: indices, their similarities, all of them."""
        every = cosine_similarity(vec.transform([question]), mat)[0]
        top = np.argsort(every)[::-1][:k]
        return top, every[top], every

    def run(where, question, vec, mat, k):
        """Call top_k_chunks on copies; give back (indices, similarities), or stop with a plain message."""
        vec_copy = copy.deepcopy(vec)              # a copy, so a slip cannot change the lab's vectorizer
        try:
            got = top_k_chunks(question, vec_copy, mat.copy(), k)
            refitted = vec_copy.vocabulary_ != vec.vocabulary_
        except Exception as err:
            refitted = getattr(vec_copy, "vocabulary_", None) != vec.vocabulary_
            if refitted:
                got, error = None, None
            elif isinstance(err, ValueError) and ("mismatch" in str(err) or "dimension" in str(err)):
                raise AssertionError(
                    f"{where}, top_k_chunks stopped with ValueError: the question's vector and the "
                    "matrix's rows have different lengths, so the question was turned into a vector "
                    "with a vocabulary of its own. Use vectorizer.transform([question]).") from None
            elif isinstance(err, IndexError):
                raise AssertionError(
                    f"{where}, top_k_chunks stopped with IndexError: {err}. Does it use a fixed number "
                    f"of chunks ({len(chunks)}, this lab's) instead of the rows of the matrix it is "
                    "given? matrix.shape[0] is the number of rows.") from None
            else:
                raise AssertionError(f"{where}, top_k_chunks stopped with {type(err).__name__}: "
                                     f"{str(err).rstrip('.')}.") from None
        assert not refitted, (
            f"{where}, top_k_chunks fitted the vectorizer again, on the question, so its vocabulary "
            "changed. Use vectorizer.transform([question]), which keeps the vocabulary learned from "
            "the chunks.")
        assert got is not None, "top_k_chunks gives back nothing: use return, not print."
        if isinstance(got, str) or (isinstance(got, (list, tuple)) and got and isinstance(got[0], str)):
            raise AssertionError(f"{where}, top_k_chunks gives back text. Give back the chunks' "
                                 "positions (indices) and their similarities.")
        if not (isinstance(got, tuple) and len(got) == 2):
            flat = np.asarray(got)
            if flat.ndim == 1 and flat.dtype.kind in "iu":
                raise AssertionError(f"{where}, top_k_chunks gives back only indices. Give back two "
                                     "arrays: the indices and their similarities.")
            if flat.ndim == 1 and flat.dtype.kind == "f":
                raise AssertionError(f"{where}, top_k_chunks gives back one array of numbers. Give "
                                     "back two arrays: the indices and their similarities.")
            raise AssertionError(f"{where}, top_k_chunks gives back a {type(got).__name__}, not two "
                                 "arrays. End it with return indices, similarities.")
        idx, sim = np.asarray(got[0]), np.asarray(got[1])
        if idx.dtype.kind == "f" and sim.dtype.kind in "iu":
            raise AssertionError(f"{where}, top_k_chunks gives back (similarities, indices). Give the "
                                 "indices first, then the similarities.")
        assert idx.dtype.kind in "iu", (
            f"{where}, the first thing top_k_chunks gives back holds {idx.dtype} values, not chunk "
            "positions (whole numbers).")
        assert idx.ndim == 1 and sim.ndim == 1, (
            f"{where}, top_k_chunks gives back arrays of shape {idx.shape} and {sim.shape}. Give back "
            "two flat arrays of k entries: a table with one row needs its row taken out, with [0].")
        return idx, sim

    def compare(where, question, vec, mat, k):
        """Stop with a message naming the slip, unless top_k_chunks matches the fixed pipeline."""
        idx, sim = run(where, question, vec, mat, k)
        ref_idx, ref_sim, every = lab_top_k(question, vec, mat, k)
        n = mat.shape[0]
        assert len(idx) != n or n == k, (
            f"{where}, top_k_chunks gives back all {n} chunks. Keep only the k most similar.")
        if len(idx) != k:
            assert not (len(idx) == 3 and k != 3), (
                f"{where}, top_k_chunks gives back 3 chunks. Does it use k, or the number 3?")
            assert abs(len(idx) - k) != 1, (
                f"{where}, top_k_chunks gives back {len(idx)} chunks: one too "
                f"{'many' if len(idx) > k else 'few'}. a[:k] keeps the first k entries of a.")
            raise AssertionError(f"{where}, top_k_chunks gives back {len(idx)} chunks.")
        assert len(sim) == len(idx), (
            f"{where}, top_k_chunks gives back {len(idx)} indices and {len(sim)} similarities. Give "
            "back the similarities of the k chunks it keeps, in the same order: pick them out with "
            "the indices.")
        least = np.argsort(every)[:k]
        assert not np.array_equal(idx, least[::-1]), (
            f"{where}, top_k_chunks gives back the {k} least similar chunks, most similar of them "
            "first: [:k] kept the k smallest before [::-1] reversed them. Reverse first, then keep k.")
        q_dense = vec.transform([question]).toarray()[0]
        dots = mat.toarray() @ q_dense
        if not np.allclose(dots[ref_idx], ref_sim):          # rows not scaled to length 1
            by_dot = np.argsort(dots)[::-1][:k]
            assert not (np.array_equal(idx, by_dot) and np.allclose(sim, dots[by_dot])), (
                f"{where}, top_k_chunks gives the chunks with the largest dot products. These rows "
                "are not scaled to length 1, so the dot product is not the cosine similarity: divide "
                "it by the product of the two lengths.")
        assert not np.array_equal(idx, least), (
            f"{where}, top_k_chunks gives back the {k} least similar chunks. np.argsort sorts from the "
            "smallest value to the largest: reverse it before keeping k.")
        if not np.array_equal(idx, ref_idx):
            if np.array_equal(idx, ref_idx[::-1]):
                raise AssertionError(f"{where}, top_k_chunks gives back the right chunks, least "
                                     "similar first. Put the most similar first.")
            if set(idx.tolist()) == set(ref_idx.tolist()):
                raise AssertionError(f"{where}, top_k_chunks gives back the right chunks in the wrong "
                                     "order. Sort them by similarity, most similar first.")
            if np.array_equal(idx, np.arange(k)):
                raise AssertionError(f"{where}, top_k_chunks gives back the first {k} chunks of the "
                                     "matrix, not the most similar. Sort by similarity first.")
            raise AssertionError(f"{where}, top_k_chunks gives back chunks {idx.tolist()}; the fixed "
                                 f"pipeline retrieves {ref_idx.tolist()}. Check the similarity, then "
                                 "the sort.")
        if not np.allclose(sim, ref_sim, rtol=1e-9, atol=1e-12):
            q_vec = vec.transform([question]).toarray()[0]
            distance = np.linalg.norm(mat.toarray()[ref_idx] - q_vec, axis=1)
            assert not np.allclose(sim, distance), (
                f"{where}, top_k_chunks gives the right chunks, with the distances between the "
                "vectors as their similarities. Give back the cosine similarity: the dot product "
                "divided by the product of the two lengths.")
            assert not np.allclose(sim, 1 - ref_sim), (
                f"{where}, top_k_chunks gives the right chunks, with 1 minus the cosine similarity, a "
                "distance. Give back the similarity itself.")
            for digits in range(1, 6):
                assert not np.allclose(sim, np.round(ref_sim, digits), rtol=0, atol=1e-12), (
                    f"{where}, top_k_chunks rounds the similarities to {digits} decimals. Give them "
                    "back unrounded: the printing rounds them.")
            raise AssertionError(
                f"{where}, top_k_chunks gives the right chunks with similarities {np.round(sim, 4)}; "
                f"the fixed pipeline has {np.round(ref_sim, 4)}. Compute the cosine similarity: the "
                "dot product divided by the product of the two lengths.")
        return idx, sim

    # 1. the question above, then two new questions, on this lab's chunks
    first_idx, first_sim = compare("On the question above, with k=3", QUESTION, vectorizer, matrix, 3)
    compare("On a question about unemployment, with k=5",
            "What happened to the unemployment rate and job gains during 2023?", vectorizer, matrix, 5)
    compare("On a question about June, with k=2",
            "Why did the Committee hold the target range steady in June 2023?", vectorizer, matrix, 2)
    # 2. a second set of chunks: 60 words each, overlapping by 15, with a vectorizer of its own
    check_chunks = build_chunks(size=60, overlap=15)
    check_vectorizer = TfidfVectorizer(stop_words="english")
    check_matrix = check_vectorizer.fit_transform(check_chunks)
    compare(f"On a second set of {len(check_chunks)} chunks, with k=4",
            "Was the policy rate at or near its peak by the end of the year?",
            check_vectorizer, check_matrix, 4)
    # 3. this lab's chunks with rows left unscaled, where a dot product alone is not the cosine
    raw_vectorizer = TfidfVectorizer(stop_words="english", norm=None)
    raw_matrix = raw_vectorizer.fit_transform(chunks)
    compare("On this lab's chunks with TfidfVectorizer(norm=None), with k=3",
            "What did members agree about the target range in December 2023?",
            raw_vectorizer, raw_matrix, 3)

    print(f"Passed: top_k_chunks matches the fixed pipeline on five runs. On the question above it "
          f"gives chunks {first_idx.tolist()} with similarities {first_sim.round(4).tolist()}.")

---

## AI-Assisted Expansion: top_k_chunks written by AI, your prompt, checked by you

You have written a retrieval function and checked it. Now ask an AI to write the same function from
a prompt you write, and compare its numbers with yours. A function this short is one an AI writes in
seconds, and it makes every choice your prompt leaves out: how to compute the similarity, how to
sort, what to give back and in what form. Some of those choices change no number, some change one in
the fourth decimal, and some change which chunks come back, which is why the check prints every
number. No prompt is given: you write it. About 25 minutes.

**The target.** The AI's function must give back the same chunks and the same similarities as your
`top_k_chunks`, on new questions, on a second set of chunks and on unscaled rows. On the question above with k=3, that
is chunks [28, 6, 12] with similarities 0.3136, 0.3068 and 0.2318.

**What the check calls.** `ai_top_k_chunks(question, vectorizer, matrix, k)`: the same arguments as
yours, in the same order, giving back the same two things. Name the function `ai_top_k_chunks` in
your prompt, or rename it when you paste it, so that it does not replace yours.

**What a good prompt states.** Your own function already decides each of these:
- the function's name, its arguments in order, and what each one is: what kind of object the
  vectorizer and the matrix are, and what was fitted on what;
- what to compute, and how: the question's vector, and the similarity;
- what to give back: which two things, in which order, sorted how, and as what type;
- what to leave out (example data, a vectorizer of its own, printing), and a comment on each line.

**What to do**
1. Write your round 1 prompt in the text cell below. Send it, exactly as written, in a new chat to
   Claude ([claude.northeastern.edu](https://claude.northeastern.edu)), or to Colab's Gemini with
   the `ask_ai` helper in the next cell.
2. Read the reply before you run anything. Paste only the function into the round 1 cell, and run
   the check below it.
3. If a line says `NO` or `missing`, or the function stops with an error, find the choice in the
   function that caused it, state that choice in a round 2 prompt, and paste and check the new
   function in the round 2 cells. Stop when every line says `yes` and you can explain each line, or
   when you decide to stop.
4. If round 1 is `Passed`, skip round 2. Instead, in Critique below, name a choice
   the AI made that your prompt never stated, and write the line you would add to your prompt to
   state it.
5. Write the short comparison under the round 2 check.

**Reading the check.** For each of five runs (the question above, two new questions, the second set
of chunks, and this lab's chunks with unscaled rows) it prints one line per rank: your chunk and the AI's, `same?`, your similarity and the
AI's, `same?`, and the % apart. Your function and the AI's run on the same vectors in the same
session, so a correct function agrees in every printed digit. There is no allowed gap: a line a
fraction of a percent apart says `NO`, and it means the function did something different, even if
that something is only rounding. The check does not say why a line differs; you find that out by reading
the AI's code. A function that gives the right numbers in another shape (a dictionary, a list of
pairs, the two arrays the other way round) is read and shown, but not passed: the target is the same
two things your function gives back.

**Colab's AI from code.** In Colab, the `google.colab.ai` module sends a prompt to one of Google's
Gemini models and gives back the reply as text: free, with no key. It is in public preview, so
Google may limit how many prompts you send. The next cell wraps it in `ask_ai(prompt)`, which you
can call in a new cell, as `print(ask_ai("""...your prompt..."""))`; text in triple quotes can run
over several lines. Outside Colab, or when the AI cannot answer (a usage limit, a network error),
`ask_ai` prints the prompt instead and gives back `None`: copy the prompt into Gemini, Claude or
ChatGPT, and paste the reply where the lab asks. `try:` runs the lines under it; if one of them
raises an error, the lines under `except` run instead. No check in this lab calls the AI: the checks
look at your code and at what you recorded.

In [ ]:
# GIVEN — run as-is
# ask_ai: a prompt to Colab's built-in Gemini, and its reply; anywhere else, the prompt printed to copy
def ask_ai(prompt):
    """Gemini's reply to prompt, through google.colab.ai; or None, after printing the prompt to copy."""
    try:
        from google.colab import ai          # exists only inside Colab
        return ai.generate_text(prompt)
    except Exception as err:                  # not in Colab, a usage limit, or any other failure
        print(f"[Colab's AI did not answer ({type(err).__name__}). Copy the prompt below into a chat "
              "AI, and paste its reply where the lab asks.]")
        print(prompt)
        return None


print("ask_ai is ready.")

In [ ]:
# GIVEN — run as-is (you need not follow this code; the cells below use it)
# The check, as a function: the cells below run it once for each round.
# It saves the lab's chunks, matrix and vocabulary now, before anything is pasted.
import copy

saved_chunks, saved_rows, saved_vocab = list(chunks), matrix.toarray(), dict(vectorizer.vocabulary_)
other_chunks = build_chunks(size=60, overlap=15)        # the second set of chunks: 60 words, overlap 15
other_vectorizer = TfidfVectorizer(stop_words="english")
other_matrix = other_vectorizer.fit_transform(other_chunks)
RUNS = [("the question above", QUESTION, vectorizer, matrix, 3),
        ("a new question", "What happened to the unemployment rate and job gains during 2023?",
         vectorizer, matrix, 5),
        ("a new question", "Why did the Committee hold the target range steady in June 2023?",
         vectorizer, matrix, 2),
        (f"{len(other_chunks)} chunks of 60 words",
         "Was the policy rate at or near its peak by the end of the year?",
         other_vectorizer, other_matrix, 4)]
raw_vectorizer = TfidfVectorizer(stop_words="english", norm=None)     # rows left unscaled
RUNS.append(("this lab's chunks, rows unscaled",
             "What did members agree about the target range in December 2023?",
             raw_vectorizer, raw_vectorizer.fit_transform(chunks), 3))


def lab_data_unchanged():
    """True if chunks, matrix and the vocabulary still hold what the fixed pipeline made."""
    # try runs the lines under it; if they raise an error (say matrix is gone), except runs instead
    try:
        return (chunks == saved_chunks and np.array_equal(matrix.toarray(), saved_rows)
                and vectorizer.vocabulary_ == saved_vocab)
    except Exception:
        return False


def as_pair(result):
    """What a function gave back, as (indices, similarities, a note on how it was read), or a message."""
    if isinstance(result, dict):                                   # {index: similarity}
        return list(result), list(result.values()), "a dictionary, read as its keys and its values"
    if isinstance(result, (tuple, list)) and result and all(       # [(index, similarity), ...]
            isinstance(pair, (tuple, list)) and len(pair) == 2 and isinstance(pair[0], (int, np.integer))
            and isinstance(pair[1], (float, np.floating)) for pair in result):
        return [p[0] for p in result], [p[1] for p in result], "(index, similarity) pairs"
    if isinstance(result, (tuple, list)) and len(result) == 2 and all(
            np.ndim(part) == 1 for part in result):
        first, second = np.asarray(result[0]), np.asarray(result[1])
        if first.dtype.kind == "f" and second.dtype.kind in "iu":
            return list(second), list(first), "(similarities, indices), read the other way round"
        return list(first), list(second), ""
    return f"gave back an object of type {type(result).__name__}; the check needs (indices, similarities)"


def show_rank(rank, mine_i, ai_i, mine_s, ai_s):
    """One line: your chunk and the AI's, your similarity and the AI's, and how far apart."""
    try:
        ai_i, ai_s = int(ai_i), float(ai_s)
    except (TypeError, ValueError):
        print(f"  {rank:>4}{mine_i:>7}{str(ai_i)[:12]:>14}{'NO':>7}   not a chunk position and a number")
        return False
    same_i = "yes" if ai_i == mine_i else "NO"
    # rtol=1e-9: equal to about 9 significant digits, so only rounding inside the computer passes
    same_s = "yes" if np.isclose(mine_s, ai_s, rtol=1e-9, atol=0) else "NO"
    pct = 100 * (ai_s - mine_s) / abs(mine_s) if mine_s else float("nan")
    # sign and 2 significant digits, so a tiny gap prints like +9.6e-06 and 1.37 as +1.4
    apart = f"{pct:+.2g}" if abs(pct) < 100 else f"{pct:+.0f}"
    print(f"  {rank:>4}{mine_i:>7}{ai_i:>7}{same_i:>7}{mine_s:>10.4f}{ai_s:>10.4f}{same_s:>7}{apart:>11}%")
    return same_i == same_s == "yes"


def check_ai_top_k(fn):
    """Run the AI's ai_top_k_chunks beside your top_k_chunks on the five runs."""
    if not lab_data_unchanged():
        print("The pasted code changed chunks, matrix or vectorizer. Paste only the function, then "
              "rerun the notebook from the top.")
        return
    if "top_k_chunks" not in globals():
        print("Write top_k_chunks (the required function above) first: this check compares the AI's function "
              "with yours.")
        return
    # callable(x) is True when x is a function that can be called
    if not callable(fn):
        print("ai_top_k_chunks should be a function. Paste the function the AI wrote.")
        return
    all_yes = True                          # stays True only if every line of every run says yes
    shapes = set()                          # how the AI's results were read, when not as two arrays
    for label, question, vec, mat, k in RUNS:
        print(f"\n{label}: \"{question}\", k={k}")
        try:
            mine = top_k_chunks(question, copy.deepcopy(vec), mat.copy(), k)
        except Exception as err:
            print(f"  Your top_k_chunks stopped with an error. {type(err).__name__}: {err}")
            all_yes = False
            continue
        given = copy.deepcopy(vec)          # a copy, so the AI's function cannot change the lab's
        try:
            result = fn(question, given, mat.copy(), k)
        except Exception as err:
            print(f"  The AI's function stopped with an error. {type(err).__name__}: {err}")
            all_yes = False
            continue
        finally:
            if getattr(given, "vocabulary_", None) != vec.vocabulary_:
                print("  The AI's function changed the vectorizer it was given: its vocabulary is "
                      "different after the call.")
                all_yes = False
        read = as_pair(result)
        if isinstance(read, str):
            print(f"  The AI's function {read}.")
            all_yes = False
            continue
        ai_idx, ai_sim, note = read
        if note:
            print(f"  the AI's function gave back {note}")
            shapes.add(note)
        print(f"  {'rank':>4}{'yours':>7}{'AI':>7}{'same?':>7}{'yours':>10}{'AI':>10}{'same?':>7}{'% apart':>12}")
        for rank in range(max(len(mine[0]), len(ai_idx))):
            if rank >= len(ai_idx):
                print(f"  {rank + 1:>4}{mine[0][rank]:>7}{'missing':>14}")
                all_yes = False
            elif rank >= len(mine[0]):
                print(f"  {rank + 1:>4}{'-':>7}{str(ai_idx[rank]):>7}{'NO':>7}   an extra chunk")
                all_yes = False
            else:
                all_yes &= show_rank(rank + 1, mine[0][rank], ai_idx[rank], mine[1][rank], ai_sim[rank])
    if all_yes and shapes:
        print(f"\nEvery number agrees, but the AI's function gave back {' and '.join(sorted(shapes))}. "
              "The target is the same two things your top_k_chunks gives back: (indices, "
              "similarities). Not passed.")
    elif all_yes:
        print("\nPassed: every line says yes, on all five runs.")
    else:
        print("\nIf a line says NO or missing, find out why before you trust the function.")

**Your prompt, round 1.** Write it here, then send it exactly as written.

*Your prompt here:*

In [ ]:
# Paste the ai_top_k_chunks function from round 1 here, then run the check below.
# Paste only the function: if the AI also wrote example data or a vectorizer of its own, leave it out.

In [ ]:
# CHECK — round 1
# To rerun this check, rerun the round 1 paste cell first: it checks whatever was pasted last
if "ai_top_k_chunks" not in globals():
    print("Paste the AI's ai_top_k_chunks function in the cell above first, keeping that name, "
          "and run that cell.")
else:
    ai_top_k_chunks_r1 = ai_top_k_chunks      # kept, so the round 2 check can tell the two apart
    check_ai_top_k(ai_top_k_chunks_r1)

**Your prompt, round 2** (if round 1 showed a `NO`, a `missing` or an error). Keep what worked in
round 1, and state what the AI had to choose on its own.

*Your prompt here:*

In [ ]:
# Paste the ai_top_k_chunks function from round 2 here, then run the check below.
# Paste only the function: if the AI also wrote example data or a vectorizer of its own, leave it out.

In [ ]:
# CHECK — round 2
# "is" asks whether two names hold the very same function: if so, nothing new was pasted
if "ai_top_k_chunks" not in globals() or ai_top_k_chunks is globals().get("ai_top_k_chunks_r1"):
    print("Paste the AI's round 2 ai_top_k_chunks function in the cell above first, keeping that "
          "name, and run that cell. If round 1 was Passed, you can skip this round.")
else:
    check_ai_top_k(ai_top_k_chunks)

**Compare.** In two or three sentences: what does the AI's last function do differently from yours
(a library call, a step, the type it gives back), even where every number agrees? Quote one line of
each.

*Your answer here:*

---

## AI-Assisted Expansion, continued: the same questions with and without the sources

Now the last step of the pipeline, generation. The lab asks Gemini two questions about 2023 twice:
once with the four retrieved chunks in the prompt and the rule to answer from them only, and once
directly, with nothing but the question. You judge each reply against the passages.
About 15 minutes.

**Reading the code.** `rag_prompt(question, k)` retrieves the `k` most similar chunks as the fixed
pipeline did and builds the prompt from them. `RULE` is a string in triple quotes, which can run over
several lines. In an f-string, `f"...{x}..."` puts the value of `x` into the text, and `"\n"` starts
a new line, so `"\n".join(...)` puts each numbered passage on its own line. `direct_prompt(question)`
is the question alone, with the same request for length. The cell prints the RAG prompt for the
first question, so that you can see what the AI receives.

In [ ]:
# GIVEN — run as-is
# The two prompts: the question with its retrieved passages, and the question alone
QUESTIONS = {
    "Q1": "How did total PCE inflation change from the February to the December 2023 meeting?",
    "Q2": "Did the Committee lower the target range at any meeting in 2023?",
}
LENGTH = "Answer in two or three sentences."
RULE = """Answer the question using only the numbered passages below, which are excerpts from the
minutes of the 2023 FOMC meetings. Cite the passage numbers you use, like [5]. If the passages do
not answer the question, say so."""


def rag_prompt(question, k=4):
    """The positions of the k chunks most similar to question, and a prompt built from them."""
    every = cosine_similarity(vectorizer.transform([question]), matrix)[0]
    ids = np.argsort(every)[::-1][:k]
    passages = "\n".join(f"[{i}] {chunks[i]}" for i in ids)
    return ids, f"{RULE} {LENGTH}\n\nPassages:\n{passages}\n\nQuestion: {question}"


def direct_prompt(question):
    """The question alone, with the same request for length."""
    return f"{question}\n{LENGTH}"


print(rag_prompt(QUESTIONS["Q1"])[1])

**Ask.** The next cell sends each question twice, with the sources and without, and keeps the four
replies in the dictionary `replies`. In Colab it takes about 20 seconds. Outside Colab, or if the
AI cannot answer, it prints the four prompts instead: copy each into a chat AI (a new chat for each,
so that no answer sees another) and paste the replies into `PASTED`, in the cell after it.

In [ ]:
# GIVEN — run as-is
# Ask each question with the sources and without; where the AI cannot answer, the prompt is printed
retrieved, replies = {}, {}
for name, question in QUESTIONS.items():
    retrieved[name], prompt = rag_prompt(question)
    print(f"========== {name}, with the sources")
    replies[f"{name} rag"] = ask_ai(prompt)
    print(f"========== {name}, direct")
    replies[f"{name} direct"] = ask_ai(direct_prompt(question))
print("replies received:", [key for key, text in replies.items() if text])

In [ ]:
# YOUR TASK — paste a reply between the triple quotes only where ask_ai printed a prompt instead
# Leave the others empty: an empty entry keeps the reply ask_ai already received.
PASTED = {
    "Q1 rag": """ """,
    "Q1 direct": """ """,
    "Q2 rag": """ """,
    "Q2 direct": """ """,
}
for key, text in PASTED.items():
    if text.strip():                 # .strip() drops spaces at both ends, so " " counts as empty
        replies[key] = text.strip()

**Read them side by side.** The next cell prints, for each question, the four passages the RAG
prompt held and then both replies. `d.get(key)` is `d[key]`, or `None` when the key is missing, and
`a or b` gives `b` when `a` is empty or `None`.

In [ ]:
# GIVEN — run as-is
# Each question: the passages its RAG prompt held, then both replies
for name, question in QUESTIONS.items():
    print(f"========== {name}: {question}")
    for i in retrieved[name]:
        print(f"  [{i}] {chunks[i]}")
    for mode in ["rag", "direct"]:
        print(f"\n  {mode.upper()} reply:\n  {replies.get(f'{name} {mode}') or '(none yet: paste it above)'}")
    print()

**Your task.** For each question, fill in `JUDGED` below: which reply the passages support
(`"rag"`, `"direct"`, `"both"` or `"neither"`), the numbers of the passages that show it, and one
sentence on why. A reply is supported when every claim in it, numbers and dates included, is in a
passage. A reply that is right from memory but whose claims are not in the passages is not supported
by them; say so in the sentence. The check after it looks only at the form of what you recorded,
never at your judgment.

In [ ]:
# YOUR TASK — for each question: which reply the passages support, the passages, and why
# "supported" is "rag", "direct", "both" or "neither"; "passages" is a list of passage numbers
JUDGED = {
    "Q1": {"supported": "", "passages": [], "why": ""},
    "Q2": {"supported": "", "passages": [], "why": ""},
}

In [ ]:
# CHECK — the replies and your judgments are recorded. Run it; you need not read it.
# It checks the form only: every reply is there, and each judgment names its answer and its passages.
KEYS = [f"{name} {mode}" for name in QUESTIONS for mode in ["rag", "direct"]]
missing = [key for key in KEYS if not str(replies.get(key) or "").strip()]
if all(not str(entry.get("supported", "")).strip() for entry in JUDGED.values()):
    if missing:
        print(f"Not recorded yet: no reply for {', '.join(missing)}. Paste each into PASTED and run "
              "that cell, then fill in JUDGED.")
    else:
        print("Not filled in yet: fill in JUDGED above, then run this check again.")
else:
    assert not missing, (
        f"No reply is recorded for {', '.join(missing)}. Paste it into PASTED and run that cell.")
    for key in KEYS:
        name, mode = key.split()
        prompt = rag_prompt(QUESTIONS[name])[1] if mode == "rag" else direct_prompt(QUESTIONS[name])
        text = replies[key].strip()
        assert text != prompt.strip() and "Passages:\n[" not in text, (
            f"{key} holds the prompt, not the AI's reply. Paste the reply.")
    for name in QUESTIONS:
        assert replies[f"{name} rag"].strip() != replies[f"{name} direct"].strip(), (
            f"The two replies for {name} are the same text. Paste each reply into its own entry.")
    assert sorted(JUDGED) == sorted(QUESTIONS), (
        f"JUDGED has the entries {sorted(JUDGED)}; it should have one for each of {sorted(QUESTIONS)}.")
    for name, entry in JUDGED.items():
        supported = entry.get("supported")
        assert supported in ("rag", "direct", "both", "neither"), (
            f"{name}: \"supported\" is {supported!r}. Write \"rag\", \"direct\", \"both\" or \"neither\".")
        passages = entry.get("passages")
        assert isinstance(passages, (list, tuple)) and all(
            isinstance(p, (int, np.integer)) for p in passages), (
            f"{name}: \"passages\" should be a list of passage numbers, like [17, 25].")
        outside = [p for p in passages if not 0 <= p < len(chunks)]
        assert not outside, (
            f"{name}: there is no passage {outside[0]}; the passages are numbered 0 to {len(chunks) - 1}.")
        assert supported == "neither" or passages, (
            f"{name}: name the passages that support the reply you chose.")
        assert len(str(entry.get("why", "")).split()) >= 5, (
            f"{name}: \"why\" needs a sentence: the claim the passages show, or do not show.")
    print(f"Passed: all {len(KEYS)} replies are recorded, and each question has a judgment.")
    for name, entry in JUDGED.items():
        print(f"  {name}: {entry['supported']}, passages {list(entry['passages'])}")

---

## AI-Assisted Expansion, continued: check an AI-written analysis against the sources

The next cell holds a four-sentence answer to the question "How did the FOMC's policy and its view of
inflation change over 2023?". An AI wrote it for this lab, and errors were planted in it on purpose.
Some sentences are supported by the excerpts and some are not: a number can be wrong, a claim can be
missing from the sources, or a conclusion can go further than the sources allow. Check every
sentence against the passages, and correct the ones that are wrong. About 15 minutes.

**Reading the code.** `enumerate(AI_ANALYSIS, start=1)` walks through the list and numbers its
entries from 1.

In [ ]:
# GIVEN — run as-is
# An AI-written answer to "How did the FOMC's policy and its view of inflation change over 2023?"
AI_ANALYSIS = [
    "Over 2023 the Committee raised the target range for the federal funds rate four times, from "
    "4-1/2 to 4-3/4 percent in February to 5-1/4 to 5-1/2 percent in July, and then held it at that "
    "level for the rest of the year.",
    "Total PCE inflation fell from 5.0 percent over the 12 months ending in December 2022 to 3.4 "
    "percent over the 12 months ending in October 2023.",
    "The unemployment rate stayed between 3.5 and 3.8 percent all year, which shows that the rate "
    "increases had no effect on the labor market.",
    "By December, participants agreed that the Committee would begin cutting rates in March 2024.",
]
for number, sentence in enumerate(AI_ANALYSIS, start=1):
    print(f"{number}. {sentence}")

**Searching the sources.** `search(text, k)` prints the `k` chunks most similar to any text, a
question or a sentence, with their numbers and similarities, retrieved as the fixed pipeline
retrieves. It matches words, not meaning: a sentence about several meetings needs several searches,
and a passage that does not come up may still exist, so search again with the words the minutes would
use. The cell runs it on sentence 1, as an example.

In [ ]:
# GIVEN — run as-is
# search: the k chunks most similar to any text, most similar first
def search(text, k=3):
    """Print the k chunks most similar to text, with their numbers and similarities."""
    every = cosine_similarity(vectorizer.transform([text]), matrix)[0]
    for i in np.argsort(every)[::-1][:k]:
        print(f"[{i}] {every[i]:.3f}  {chunks[i]}")


search(AI_ANALYSIS[0])

**Your task.** Fill in the table under the line below, one row per sentence: whether the passages
support it (yes or no), the numbers of the passages you checked, what they say (a short quote), and,
for a sentence they do not support, a corrected sentence, or "remove" if the sources cannot support
any version of it. Run `search` in new cells as often as you need (**Insert → Code cell**).

*Your answers here:*

| Sentence | Supported? | Passages checked | What the passages say | Correction |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |

---

### Checking log

This is graded, with your prompts, `JUDGED`, the critique table and the Critique and Judgment answers below. One row
per task: what you checked, how you checked it, and what you found. A row that names the lines that
differed, how far apart they were and the line of the AI's code responsible, or the passage that
contradicts a claim, earns credit; "it worked" does not. The last row is where you stopped, and why.

| Task | What I checked | How | What I found |
|---|---|---|---|
| `ai_top_k_chunks`, round 1 | | | |
| `ai_top_k_chunks`, round 2 (write "not needed" if round 1 was Passed) | | | |
| RAG against direct, Q1 and Q2 | | | |
| The AI-written analysis | | | |
| Where I stopped | | | |

### Questions

**Critique, in your own words.** Name one choice the AI made in `ai_top_k_chunks` that your prompt never stated, quote the line where it makes it, and say whether you would keep it and why. (A choice that caused a `NO`, a `missing` or an error counts. If round 1 was `Passed`, name a choice your prompt never stated and write the prompt line that would state it.) Then name one claim in a reply to Q1 or Q2 that no passage supports, and say where you think the AI got it.

**Judgment: where you stopped.** Which of this section's three uses of AI would you trust without checking, and which not, and why? Then use your two questions: which reply did the sources change, and which question would any AI answer the same way without them? From that, say when RAG over documents like these minutes beats asking an AI directly, when it does not matter, and when you would read the documents yourself.

*Your answers here:*

**Critique:**

**Judgment:**

---

## Final Reflection

Answer in 2-3 sentences.

### R1: How do chunking parameters affect retrieval quality?

*Connect it to the bias-variance tradeoff from Ch 15, and use the analyst's 8-word chunks and the lab's 40-word ones.*

*Your answer here:*

---

## Submission

1. **Your portfolio:** the live Vercel URL and the link to its GitHub repository.
2. **This notebook**, with every cell run and every text cell filled in. In Colab use **File →
   Download → Download .ipynb**, upload that file to the Canvas assignment, and paste the portfolio
   URL and the repository link in the **Comments** box beside the upload. Then click **Submit**. Do
   not use the Website URL tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

Push your last portfolio changes before you submit:

```bash
cd econ-portfolio
git add .
git commit -m "Lab 25: portfolio"
git push origin main
```